<a href="https://colab.research.google.com/github/lalawee/special-octo-barnacle/blob/main/Task2_SARSA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""Task2_SARSA.ipynb

#OpenAI Gym - Frozen Lake (SARSA)
"""

import numpy as np
import gym
from IPython.display import clear_output
import time

# cause the new version of numpy has removed np.bool8 type
# and replace it with np.bool_
# gym still uses np.bool8
np.bool8 = np.bool_

#
# Parameters
#
EPISODES = 10000
GAMMA = 0.9       # discount factor
ALPHA = 0.1       # learning rate
EPS = 1.0         # initial epsilon
MIN_EPS = 0.01    # minimum epsilon
DECAY = 0.999     # epsilon decay rate per episode

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [2]:
"""## Setup Gym Env"""

#
# The reward is 1 if you reach the Goal and 0 otherwise.
# Landing on a hole will terminate the game
# is_slippery=False -> it's deterministic
# render_mode='ansi' refers to display on text-based console
#
# For details: https://gymnasium.farama.org/environments/toy_text/frozen_lake/
#
env = gym.make('FrozenLake-v1', new_step_api=True, render_mode='ansi', is_slippery=False)

print("No of states:", env.observation_space.n)
print("No of actions:", env.action_space.n)

print(env.observation_space) # meaning values are 0 to 15
print(env.action_space) # meaning values are 0 to 3

"""# Helper to extract state as int"""

def get_state(obs):
    """Extract integer state from whatever env.reset() returns."""
    if isinstance(obs, tuple):
        return int(obs[0])
    return int(obs)

"""# Epsilon-greedy action selection"""

def epsilon_greedy(state, epsilon):
    if np.random.random() < epsilon:
        return env.action_space.sample()  # explore
    else:
        return np.argmax(Q[state, :])     # exploit


No of states: 16
No of actions: 4
Discrete(16)
Discrete(4)


In [3]:
"""# Part 1: Training with SARSA"""

# Initialize Q-table with zeros: 16 states x 4 actions
Q = np.zeros((env.observation_space.n, env.action_space.n))

total_goal = 0
epsilon = EPS

for episode in range(EPISODES):
    state = get_state(env.reset())
    truncated = False
    terminated = False

    # SARSA: choose initial action using epsilon-greedy
    action = epsilon_greedy(state, epsilon)

    while not (truncated or terminated):
        result = env.step(action)
        next_state = int(result[0])
        reward = result[1]
        terminated = result[2]
        truncated = result[3]

        # SARSA: choose next action using epsilon-greedy (on-policy)
        next_action = epsilon_greedy(next_state, epsilon)

        # SARSA update (on-policy): uses the actual next action chosen
        # If terminated, there is no future value
        if terminated:
            td_target = reward
        else:
            td_target = reward + GAMMA * Q[next_state, next_action]

        Q[state, action] = Q[state, action] + ALPHA * (td_target - Q[state, action])

        if reward == 1:
            total_goal += 1

        state = next_state
        action = next_action  # carry forward the chosen next action

    # Decay epsilon after each episode
    epsilon = max(MIN_EPS, epsilon * DECAY)

print(f"Training Finished. Total Goals: {total_goal}, Cumulative Success Rate: {(total_goal * 100 / EPISODES):0.1f}%")

Training Finished. Total Goals: 8802, Cumulative Success Rate: 88.0%


In [4]:
"""## Print Optimal Q-Table"""

print("Optimal Q-Table:")
print(Q)


Optimal Q-Table:
[[0.50274199 0.43571026 0.59012246 0.51146547]
 [0.50469171 0.         0.65605522 0.57672817]
 [0.53000755 0.72898739 0.53528294 0.62409543]
 [0.64187699 0.         0.22147501 0.12517929]
 [0.13975428 0.13840383 0.         0.51285005]
 [0.         0.         0.         0.        ]
 [0.         0.80999765 0.         0.61603731]
 [0.         0.         0.         0.        ]
 [0.00156427 0.         0.33758289 0.03893187]
 [0.07528926 0.38959672 0.80108487 0.        ]
 [0.7065902  0.89999978 0.         0.70544984]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.56061848 0.8969965  0.25244933]
 [0.79200108 0.89944017 1.         0.77667188]
 [0.         0.         0.         0.        ]]


In [5]:
"""# Part 2: Visualizing Optimal Policy Movement"""

print("\nVisualizing Optimal Policy Movement:")
for episode in range(3):
    state = get_state(env.reset())
    terminated = False
    truncated = False
    step_count = 0

    while not (truncated or terminated):
        # Display the frame
        clear_output(wait=True)
        print(f"Episode {episode + 1}  |  Step {step_count}")
        for line in env.render():
            print(line)
        time.sleep(0.5)

        # Use greedy policy from learned Q-values
        action = np.argmax(Q[state, :])
        result = env.step(action)
        state = int(result[0])
        reward = result[1]
        terminated = result[2]
        truncated = result[3]
        step_count += 1

    # Show final frame
    clear_output(wait=True)
    print(f"Episode {episode + 1} - Final Result")
    for line in env.render():
        print(line)
    if reward == 1:
        print(f"=> Goal reached in {step_count} steps")
    else:
        print("=> Fell into a hole.")
    time.sleep(1.0)

Episode 3 - Final Result
  (Right)
SFFF
FHFH
FFFH
HFFG

=> Goal reached in 6 steps
